# 🛡️ Enterprise Credential Exposure & Risk Simulator
**Dataset:** Have I Been Pwned (HIBP) Mega-Breaches (2024–2026)  
**Author:** Oluwatimilehin Aramide

**Focus:** Threat Intelligence, Credential Stuffing Modeling, and MFA Defense Impact

# Organizational Identity Risk Analysis: Data Breaches, Credential Exposure & MFA

## Project Overview

This project examines how large-scale data breaches create security risks for organizations and their employees.

Using a dataset of 20 major data breaches added to Have I Been Pwned between 2024 and 2026, I analyzed how stolen credentials can be used to gain unauthorized access to employee accounts.

The project focuses on how cyber threats have evolved from individual website breaches to large collections of stolen login information and infostealer malware that can also capture active browser sessions.

A simulated organization of 250 employees was used to estimate employee exposure and compare account takeover risks with and without Multi-Factor Authentication (MFA).

## Project Objectives

1. **Estimate Employee Exposure:** Calculate the probability that at least one employee's email address appears in a large-scale data breach.
2. **Compare Cyber Threats:** Examine the differences between single-service breaches, credential stuffing lists, and infostealer logs.
3. **Evaluate MFA Effectiveness:** Simulate how MFA reduces the likelihood of unauthorized account access through stolen credentials.
4. **Identify Security Gaps:** Investigate how attackers can bypass traditional authentication by stealing active browser session cookies.
5. **Recommend Security Improvements:** Identify practical ways organizations can protect employee accounts and sensitive information.

## Key Findings

### 1. High Probability of Employee Exposure
The simulation estimated a 91.9% probability that at least one employee in a 250-person organization would appear in the smallest breach dataset analyzed. For larger aggregated datasets, the estimated probability reached 100%.

This demonstrates why organizations should not assume their employees have avoided exposure simply because they have not experienced a direct company breach.

### 2. Credential Aggregation Increases Risk
Individual website breaches generally create a more limited exposure compared with large collections of stolen credentials.

For example, the Synthient Credential Stuffing dataset contained approximately 2 billion records. Under the project's password-reuse assumptions, the simulation estimated up to 88 compromised accounts without MFA.

### 3. Infostealers Create Additional Security Risks
Infostealer malware can collect saved passwords, browser information, and active session cookies.

This creates an additional risk because attackers may use stolen session cookies to access accounts without completing the normal login process.

### 4. MFA Reduces Account Takeover Risks
The simulation demonstrated that implementing MFA can significantly reduce successful account takeover attempts involving stolen and reused passwords.

However, traditional MFA methods are not completely effective against session hijacking and certain advanced phishing attacks.

## Recommended Security Measures

| Priority | Recommendation | Purpose |
|---|---|---|
| P0 | Enterprise Password Vaults | Reduce browser-based password storage and improve credential management. |
| P0 | Automated Breach Monitoring | Detect exposed employee credentials and trigger appropriate security responses. |
| P1 | Device Trust | Restrict access to sensitive systems from unmanaged or non-compliant devices. |
| P1 | Continuous Access Evaluation | Detect suspicious activity and revoke compromised sessions. |
| P2 | FIDO2 / Passkeys | Strengthen authentication and reduce phishing and credential theft risks. |

## Tools & Concepts

**Security Concepts**
- Data breach analysis
- Credential stuffing
- Infostealer malware
- Multi-Factor Authentication (MFA)
- Session hijacking
- Phishing-resistant authentication

**Analytical Concepts**
- Probability modelling
- Risk estimation
- Simulation
- Comparative analysis
- Organizational exposure assessment

## Key Takeaway

The main takeaway from this project is that an organization does not need to experience a direct data breach to face security risks.

Employees may already have exposed credentials from unrelated websites, and attackers can use those credentials to target organizational systems.

MFA provides an important layer of protection, but organizations also need to protect devices, monitor exposed credentials, manage active sessions, and adopt stronger authentication methods.

This project demonstrates how data analysis and probability modelling can help translate cybersecurity threats into measurable organizational risks and practical security recommendations.

---

*Note: All exposure and account compromise figures are simulation-based estimates. Actual outcomes depend on the model assumptions, breach data quality, password reuse patterns, and security controls in place.*

In [1]:
%pip install pandas matplotlib ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd

# Load the HIBP 2024-2026 CSV
df = pd.read_csv("have-i-been-pwned-breaches-2024-2026-data.csv")

# Clean whitespace from column names if present
df.columns = df.columns.str.strip()

# Display summary table
display(df.head(10))
print(f"Total breaches loaded: {len(df)}")
print(f"Unique breach categories: {df['scan_my_shadow_type_label'].unique()}")

,breach,breach_month,added_to_hibp,accounts_millions_rounded,scan_my_shadow_type_label,source_url
0,Synthient Credential Stuffing Threat Data,Apr 2025,6 Nov 2025,2000.0,credential list,https://haveibeenpwned.com/PwnedWebsites
1,Combolists Posted to Telegram,May 2024,3 Jun 2024,361.5,credential list,https://haveibeenpwned.com/PwnedWebsites
2,ALIEN TXTBASE Stealer Logs,Feb 2025,25 Feb 2025,284.1,stealer log,https://haveibeenpwned.com/PwnedWebsites
3,Not SOCRadar,Aug 2024,9 Aug 2024,282.5,see HIBP entry,https://haveibeenpwned.com/PwnedWebsites
4,Synthient Stealer Log Threat Data,Apr 2025,21 Oct 2025,183.0,stealer log,https://haveibeenpwned.com/PwnedWebsites
5,National Public Data (flagged unverified by HIBP),Apr 2024,13 Aug 2024,134.0,single service,https://haveibeenpwned.com/PwnedWebsites
6,DemandScience by Pure Incubation,Feb 2024,13 Nov 2024,121.8,single service,https://haveibeenpwned.com/PwnedWebsites
7,Data Troll Stealer Logs,Jun 2025,13 Aug 2025,109.5,stealer log,https://haveibeenpwned.com/PwnedWebsites
8,1win,Nov 2024,3 Feb 2025,96.2,single service,https://haveibeenpwned.com/PwnedWebsites
9,Advance Auto Parts,Jun 2024,24 Jun 2024,79.2,single service,https://haveibeenpwned.com/PwnedWebsites


Total breaches loaded: 20
Unique breach categories: ['credential list' 'stealer log' 'see HIBP entry' 'single service']


In [3]:
def get_exposure_probability(accounts_millions, breach_type):
    """Estimate the probability that a random individual's credentials

    appear in a dataset of this size and classification.
    """
    label = str(breach_type).lower()

    if "credential list" in label or accounts_millions >= 500:
        # Aggregated credential dumps have massive historical coverage
        return min(0.70, 0.20 + (accounts_millions / 2000.0) * 0.50)
    elif "stealer log" in label:
        # Infostealers capture actively logged-in devices
        return min(0.35, 0.05 + (accounts_millions / 500.0) * 0.25)
    else:
        # Single service / standard breaches
        # Global consumer population ~5 billion internet users
        return min(0.15, max(0.01, accounts_millions / 5000.0))


def run_simulation(breach_name, num_employees, reuse_rate_pct):
    row = df[df["breach"] == breach_name].iloc[0]

    accounts_m = float(row["accounts_millions_rounded"])
    b_type = row["scan_my_shadow_type_label"]
    p_exposed = get_exposure_probability(accounts_m, b_type)

    reuse_rate = reuse_rate_pct / 100.0

    # 1. Probability that AT LEAST ONE company employee is in the breach
    p_team_hit = 1.0 - ((1.0 - p_exposed) ** num_employees)

    # 2. Expected number of employees exposed
    expected_exposed = round(num_employees * p_exposed)

    # 3. Compromises without MFA (exposed + reused password)
    compromised_no_mfa = round(expected_exposed * reuse_rate)

    # 4. Compromises with MFA (blocks ~99% of basic credential stuffing)
    mfa_block_rate = 0.99
    compromised_with_mfa = round(compromised_no_mfa * (1.0 - mfa_block_rate))

    return {
        "breach": breach_name,
        "type": b_type,
        "size_m": accounts_m,
        "p_team_hit": p_team_hit * 100,
        "expected_exposed": expected_exposed,
        "compromised_no_mfa": compromised_no_mfa,
        "compromised_with_mfa": compromised_with_mfa,
    }

In [4]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import clear_output, display


def update_dashboard(breach_name, num_employees, reuse_rate_pct):
    results = run_simulation(breach_name, num_employees, reuse_rate_pct)

    # Plotting comparison
    fig, ax = plt.subplots(figsize=(8, 4.5), dpi=100)

    categories = ["Exposed in Breach", "Compromised (No MFA)", "Compromised (With MFA)"]
    values = [
        results["expected_exposed"],
        results["compromised_no_mfa"],
        results["compromised_with_mfa"],
    ]
    colors = ["#f39c12", "#e74c3c", "#2ecc71"]

    bars = ax.bar(categories, values, color=colors, width=0.55)

    # Add numeric labels on top of bars
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{int(height)}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    ax.set_ylabel("Number of Employees / Accounts")
    ax.set_ylim(0, max(max(values) * 1.25, 5))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

    title_text = (
        f"Breach: {results['breach']} ({results['type']}) | Scale: {results['size_m']}M\n"
        f"Chance of Team Exposure: {results['p_team_hit']:.1f}% across {num_employees} employees"
    )
    ax.set_title(title_text, fontsize=11, fontweight="bold", pad=12)

    plt.tight_layout()
    plt.show()

    # Informational notes
    if "stealer log" in str(results["type"]).lower():
        print("⚠️ NOTE: This is an Infostealer dataset. Malware steals active session cookies,")
        print("   meaning simple SMS/App MFA can be bypassed. FIDO2 / Passkeys required.")
    else:
        prevented = results["compromised_no_mfa"] - results["compromised_with_mfa"]
        print(f"🛡️ MFA DEFENSE: Enforcing basic MFA protects ~{prevented} corporate accounts from being stuffed.")


# Create widgets
breach_dropdown = widgets.Dropdown(
    options=list(df["breach"].values),
    value=df["breach"].values[0],
    description="Breach:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)

employees_slider = widgets.IntSlider(
    value=200,
    min=10,
    max=1000,
    step=10,
    description="Employees:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)

reuse_slider = widgets.IntSlider(
    value=50,
    min=10,
    max=90,
    step=5,
    description="Password Reuse %:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="450px"),
)

interactive_plot = widgets.interactive_output(
    update_dashboard,
    {
        "breach_name": breach_dropdown,
        "num_employees": employees_slider,
        "reuse_rate_pct": reuse_slider,
    },
)

display(widgets.VBox([breach_dropdown, employees_slider, reuse_slider]), interactive_plot)

Output()

In [5]:
# Batch run across all 20 entries
company_size = 250
reuse_rate = 50

all_results = [run_simulation(name, company_size, reuse_rate) for name in df["breach"]]
summary_df = pd.DataFrame(all_results)

# Sort by expected accounts compromised
summary_df = summary_df.sort_values(by="compromised_no_mfa", ascending=False)

summary_df[[
    "breach", 
    "type", 
    "size_m", 
    "p_team_hit", 
    "compromised_no_mfa", 
    "compromised_with_mfa"
]].reset_index(drop=True)

,breach,type,size_m,p_team_hit,compromised_no_mfa,compromised_with_mfa
0,Synthient Credential Stuffing Threat Data,credential list,2000.0,100.000000,88,1
1,Combolists Posted to Telegram,credential list,361.5,100.000000,36,0
2,ALIEN TXTBASE Stealer Logs,stealer log,284.1,100.000000,24,0
3,Synthient Stealer Log Threat Data,stealer log,183.0,100.000000,18,0
4,Data Troll Stealer Logs,stealer log,109.5,100.000000,13,0
5,June 2026 Stealer Logs,stealer log,56.3,100.000000,10,0
6,"Stealer Logs, Jan 2025",stealer log,71.0,100.000000,10,0
7,Not SOCRadar,see HIBP entry,282.5,99.999952,7,0
8,National Public Data (flagged unverified by HIBP),single service,134.0,99.887662,4,0
9,DemandScience by Pure Incubation,single service,121.8,99.789913,3,0
